Every Python file is a module, and every directory of modules can become a package. The import system is how programs split across multiple files share code — it is also how the standard library and third-party libraries reach your code. Understanding how imports work, how Python finds modules, how packages are structured, and how to manage dependencies with virtual environments are essential skills for writing anything beyond a single script. This notebook covers all of it.

## What Is a Module?

A **module** is any `.py` file. When you `import` it, Python executes the file from top to bottom and makes its names available under the module's namespace. The module object is cached in `sys.modules` so subsequent imports of the same module are instant — the file is not re-executed.

Python ships with hundreds of modules in the **standard library** — `os`, `sys`, `math`, `datetime`, `json`, `pathlib`, `itertools`, `collections`, and many more. They are always available without installing anything.

In [ ]:
# Import the module — access its names with dot notation
import math

print(math.pi)              # 3.141592653589793
print(math.sqrt(144))       # 12.0
print(math.floor(3.9))      # 3

# The module is an object
print(type(math))           # <class 'module'>
print(math.__file__)        # path to the .py or .so file

In [ ]:
# Import with an alias — useful for long names or name conflicts
import datetime as dt
import collections as col

today = dt.date.today()
print(today)

counts = col.Counter(["a", "b", "a", "c", "a", "b"])
print(counts.most_common())

In [ ]:
# from ... import — bring specific names into the current namespace
from math import pi, sqrt, ceil
from pathlib import Path
from collections import defaultdict, Counter

print(pi)          # no 'math.' prefix needed
print(sqrt(256))   # 16.0

# from ... import ... as — rename on import
from datetime import datetime as DateTime
now = DateTime.now()
print(now)

In [ ]:
# Avoid 'from module import *' — it pollutes the namespace and obscures where names come from
# The ONE exception is interactive shells / REPL sessions

# Inspect what a module exports
import os.path
public_names = [name for name in dir(os.path) if not name.startswith("_")]
print(public_names[:10])   # first 10 public names

## Module Search Path — `sys.path`

When you write `import foo`, Python searches for `foo` in this order:

1. `sys.modules` cache — if already imported, return immediately.
2. Built-in modules (compiled into the interpreter: `sys`, `builtins`, …).
3. Frozen modules.
4. The directories listed in `sys.path`.

`sys.path` is built from:
- The directory of the script being run (or `""` for the current directory in the REPL).
- Directories in the `PYTHONPATH` environment variable.
- The standard library and site-packages directories.

You can append to `sys.path` at runtime, but prefer proper package installation over this.

In [ ]:
import sys

print("Python version:", sys.version)
print()
print("sys.path entries:")
for p in sys.path:
    print(f"  {p}")

print()
# Check where a module was found
import json
print("json module:", json.__file__)

In [ ]:
# sys.modules — the import cache
import sys

# After importing, the module is cached
import math
print("math" in sys.modules)      # True
print(sys.modules["math"] is math) # True — same object

# Importing again returns the cached version instantly
import math as math2
print(math is math2)               # True

## `__name__` and the Script Guard

Every module has a `__name__` attribute. When a file is **imported**, `__name__` is the module's name (e.g. `"utils"`). When a file is **run directly** as a script, `__name__` is `"__main__"`.

The `if __name__ == "__main__":` guard lets a file serve dual purpose — importable library code and runnable script — without the script-level code running when the file is imported.

In [ ]:
# Demonstrate __name__ in the current context
print(f"__name__ in this notebook: {__name__!r}")
# In a notebook or REPL: '__main__'
# When imported as a module: the module's filename without .py

# Pattern for files that are both importable and runnable:
# ─── utils.py ─────────────────────────────────────────────
# def add(a, b):
#     return a + b
#
# def main():
#     print(add(2, 3))   # runs only when executed directly
#
# if __name__ == "__main__":
#     main()
# ──────────────────────────────────────────────────────────
#
# 'python utils.py'  → main() runs
# 'import utils'     → main() does NOT run; add() is available

## Packages

A **package** is a directory that contains Python modules. It must contain an `__init__.py` file (which may be empty). The `__init__.py` runs when the package is imported and controls what the package exposes.

**Namespace packages** (PEP 420, Python 3.3+) do not need `__init__.py` — but explicit `__init__.py` files are still strongly recommended for regular packages.

```text
mypackage/
├── __init__.py        ← makes mypackage importable
├── core.py
├── utils.py
└── subpackage/
    ├── __init__.py
    └── helpers.py
```

In [ ]:
import os, sys, tempfile
from pathlib import Path

# Build a sample package in a temp directory so we can import it
TMP = Path(tempfile.mkdtemp())
sys.path.insert(0, str(TMP))   # add to import path

pkg = TMP / "mypackage"
pkg.mkdir()
(pkg / "__init__.py").write_text("""
# mypackage/__init__.py
from .core  import add, multiply
from .utils import clamp

__version__ = "1.0.0"
__all__ = ["add", "multiply", "clamp"]
""")

(pkg / "core.py").write_text("""
def add(a, b):
    """Return the sum of a and b."""
    return a + b

def multiply(a, b):
    """Return the product of a and b."""
    return a * b
""")

(pkg / "utils.py").write_text("""
def clamp(value, lo, hi):
    """Clamp value to [lo, hi]."""
    return max(lo, min(value, hi))
""")

print("Package created:", pkg)

In [ ]:
import mypackage

print(mypackage.__version__)           # 1.0.0
print(mypackage.add(3, 4))             # 7
print(mypackage.clamp(15, 0, 10))      # 10

# Import submodules directly
from mypackage.core import multiply
print(multiply(6, 7))                  # 42

# Inspect the package
print(mypackage.__file__)              # path to __init__.py
print(mypackage.__all__)               # ['add', 'multiply', 'clamp']

## Relative Imports

Inside a package, modules can import from each other using **relative imports** — a leading dot means "current package", two dots mean "parent package".

```python
# Inside mypackage/core.py:
from .utils import clamp       # sibling module
from ..other import something  # parent package's module
```

Relative imports work only inside packages (files that are part of a package and imported, not run directly). They are more robust than absolute imports inside a package because they do not depend on the package's install location.

In [ ]:
# Build a package with a subpackage to demonstrate relative imports
sub = pkg / "subpackage"
sub.mkdir()

(sub / "__init__.py").write_text("""
from .helpers import format_result
""")

(sub / "helpers.py").write_text("""
from ..utils import clamp   # relative import — go up one level to mypackage, then import utils

def format_result(value, lo=0, hi=100):
    clamped = clamp(value, lo, hi)
    return f"Result: {clamped} (clamped to [{lo}, {hi}])"
""")

# Reload mypackage so it picks up the subpackage
import importlib
importlib.invalidate_caches()

from mypackage.subpackage import format_result
print(format_result(150))    # Result: 100 (clamped to [0, 100])
print(format_result(-5))     # Result: 0   (clamped to [0, 100])

## `__all__` — Controlling Exports

Defining `__all__` in a module (or `__init__.py`) serves two purposes:

1. **Documents the public API** — makes explicit which names are intended for external use.
2. **Controls `from module import *`** — only names in `__all__` are imported by a star import.

If `__all__` is not defined, a star import imports every name that does not start with an underscore. Defining `__all__` is good practice for any module you share.

In [ ]:
# Demonstrate __all__
(TMP / "shapes.py").write_text("""
import math

__all__ = ["Circle", "Rectangle"]  # Triangle is intentionally private

class Circle:
    def __init__(self, r): self.r = r
    def area(self): return math.pi * self.r ** 2

class Rectangle:
    def __init__(self, w, h): self.w, self.h = w, h
    def area(self): return self.w * self.h

class _InternalHelper:   # leading _ also signals private
    pass

def _private_helper():
    pass
""")

import importlib, shapes
print(shapes.__all__)      # ['Circle', 'Rectangle']

c = shapes.Circle(5)
print(f"{c.area():.2f}")

# dir() shows everything; __all__ shows the intended public surface
print([n for n in dir(shapes) if not n.startswith("_")])

## The Standard Library — Key Modules

Python's standard library is vast. These are the modules every Python programmer reaches for regularly:

In [ ]:
# os — operating system interfaces
import os
print(os.getcwd())                 # current working directory
print(os.environ.get("HOME", ""))  # read environment variable
print(os.cpu_count())              # number of CPUs

In [ ]:
# sys — Python runtime information
import sys
print(sys.version_info)            # (3, 10, ...)
print(sys.platform)                # 'darwin', 'linux', 'win32'
print(sys.argv)                    # command-line arguments
# sys.exit(0)                      # exit with code 0

In [ ]:
# datetime — dates, times, and durations
from datetime import date, datetime, timedelta

today    = date.today()
now      = datetime.now()
birthday = date(1990, 6, 15)
age      = (today - birthday).days // 365

print(today, now)
print(f"Age: ~{age} years")
print(now + timedelta(days=30))    # 30 days from now

In [ ]:
# itertools — combinatorial and iterator utilities
import itertools

# chain — flatten iterables
combined = list(itertools.chain([1,2], [3,4], [5,6]))
print(combined)

# islice — lazy slice of an iterator
first5 = list(itertools.islice(itertools.count(10), 5))  # 10,11,12,13,14
print(first5)

# product — Cartesian product
pairs = list(itertools.product(["A","B"], [1,2,3]))
print(pairs)

# groupby — group consecutive equal elements
data = [("dev","Alice"),("dev","Bob"),("ops","Carol"),("ops","Dave")]
for dept, members in itertools.groupby(data, key=lambda x: x[0]):
    print(dept, list(m[1] for m in members))

In [ ]:
# functools — higher-order function utilities
import functools

# lru_cache — memoisation
@functools.lru_cache(maxsize=None)
def fib(n):
    if n < 2: return n
    return fib(n-1) + fib(n-2)

print([fib(i) for i in range(10)])
print(fib.cache_info())

# reduce — fold a sequence to a single value
product = functools.reduce(lambda a, b: a * b, [1,2,3,4,5])
print(product)   # 120

# partial — pre-fill arguments
add10 = functools.partial(int, base=16)   # always parse as hex
print(add10("ff"))   # 255

In [ ]:
# collections — specialised container types
from collections import Counter, defaultdict, deque, OrderedDict, namedtuple

# deque — O(1) append and pop from both ends
q = deque([1, 2, 3])
q.appendleft(0)
q.append(4)
print(q)                  # deque([0, 1, 2, 3, 4])
print(q.popleft())        # 0  — O(1), unlike list.pop(0) which is O(n)

# namedtuple — lightweight immutable record
Point = namedtuple("Point", ["x", "y"])
p = Point(3, 4)
print(p, p.x, p[1])

In [ ]:
# re — regular expressions
import re

text = "Call us at 555-1234 or 555-5678 before 9pm"

# findall — list of all matches
phones = re.findall(r"\d{3}-\d{4}", text)
print(phones)   # ['555-1234', '555-5678']

# sub — replace matches
redacted = re.sub(r"\d{3}-\d{4}", "XXX-XXXX", text)
print(redacted)

# Compile a pattern for repeated use
email_re = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
emails = email_re.findall("alice@example.com and bob@work.io")
print(emails)

In [ ]:
# logging — structured application logging
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)-8s %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)

logger = logging.getLogger("myapp")
logger.debug("Starting up")
logger.info("Server listening on :8080")
logger.warning("Memory usage above 80%%")
logger.error("Connection to database failed")

## Third-Party Packages and `pip`

The Python Package Index (PyPI) hosts hundreds of thousands of third-party packages. `pip` is the standard installer.

```bash
# Install a package
pip install requests

# Install a specific version
pip install requests==2.31.0

# Install from a requirements file
pip install -r requirements.txt

# List installed packages
pip list

# Show details about a package
pip show requests

# Save current environment to requirements.txt
pip freeze > requirements.txt

# Uninstall
pip uninstall requests
```

A `requirements.txt` pins exact versions — essential for reproducible builds:
```
requests==2.31.0
pydantic==2.5.0
httpx>=0.25,<0.26
```

## Virtual Environments

A **virtual environment** is an isolated Python installation — its own `site-packages`, its own `pip`. Different projects can have different (even conflicting) dependency versions without interfering with each other or with the system Python.

**Always use a virtual environment for every project.** It is the single most important habit for avoiding dependency chaos.

```bash
# Create a virtual environment named .venv in the current directory
python -m venv .venv

# Activate it
source .venv/bin/activate      # macOS / Linux
.venv\Scripts\activate         # Windows

# Your prompt changes to show (.venv)
# pip install now installs into .venv, not the system
pip install requests

# Deactivate when done
deactivate
```

Always add `.venv/` (or `venv/`) to your `.gitignore` — never commit the virtual environment directory.

## Import Patterns

A few patterns that come up in real codebases:

In [ ]:
# Conditional import — handle optional or platform-specific dependencies
try:
    import ujson as json   # faster JSON library if available
except ImportError:
    import json            # fall back to standard library

print(json.dumps({"status": "ok"}))

In [ ]:
# Lazy import — defer expensive imports until they are actually needed
# Speeds up startup time for CLIs and large applications

def process_image(path):
    import PIL.Image   # only imported when this function is called
    img = PIL.Image.open(path)
    return img.size

# PIL is only imported if process_image() is actually called
# Applications that never call it never pay the import cost

In [ ]:
# importlib — programmatic import when the module name is a string
import importlib

module_name = "math"
mod = importlib.import_module(module_name)
print(mod.pi)   # 3.141592653589793

# Useful for plugin systems where module names come from config files
plugins = ["json", "csv", "os.path"]
loaded = {name: importlib.import_module(name) for name in plugins}
print({name: mod.__name__ for name, mod in loaded.items()})

In [ ]:
# Typical __init__.py for a well-structured package:
# - Re-exports the public API so callers import from the package, not submodules
# - Sets __version__
# - Defines __all__

example_init = '''
"""mylib — does useful things."""

__version__ = "1.2.0"
__author__  = "Alice"

from .client  import Client
from .models  import User, Order
from .errors  import MyLibError, NotFoundError

__all__ = ["Client", "User", "Order", "MyLibError", "NotFoundError"]
'''
# With this __init__.py, users write:
#   from mylib import Client, User
# rather than:
#   from mylib.client import Client
#   from mylib.models import User
print(example_init)

## Summary

- A **module** is any `.py` file. Import it with `import name`, `from name import x`, or `import name as alias`. Modules are executed once and cached in `sys.modules`.
- **`sys.path`** is the list of directories Python searches when importing. It starts with the script's directory and includes standard library and site-packages paths.
- **`__name__`** is `"__main__"` when a file is run directly, and the module's name when imported. The `if __name__ == "__main__":` guard separates library code from script code.
- A **package** is a directory with an `__init__.py`. The init file runs on import and should re-export the package's public API. Use `from .sibling import x` (relative imports) for cross-module imports within a package.
- **`__all__`** is a list of names that `from module import *` exports. It also documents the public API. Define it in every module you share.
- Key standard library modules: `os`, `sys`, `math`, `datetime`, `pathlib`, `json`, `csv`, `re`, `itertools`, `functools`, `collections`, `logging`, `io`, `tempfile`, `shutil`.
- **`pip install package`** installs from PyPI. Pin versions in `requirements.txt` for reproducible builds (`pip freeze > requirements.txt`).
- **Virtual environments** (`python -m venv .venv`) isolate each project's dependencies. Always use one per project. Never commit the `.venv` directory.
- **Conditional import** (`try: import fast_lib except ImportError: import stdlib_lib`) handles optional dependencies. **Lazy import** (import inside a function) defers cost until needed. **`importlib.import_module`** imports by string name for dynamic plugin systems.